# Run the experiment on a Kaggle T4

This notebook runs the whole pipeline on a GPU and produces every number in
`results/`. Everything else in this repo runs on a laptop; only the extraction
stage needs CUDA.

**Before you start**

1. Settings → Accelerator → **GPU T4 x2** (one is enough; NF4 keeps the 7B under 16 GB).
2. Settings → Internet → **On** (needed to fetch the model and the dataset).
3. Get this repo into the session — either set `REPO_URL` below to your GitHub
   remote, or upload the repo as a Kaggle Dataset and set `INPUT_DIR`.

Expected wall clock at `n_examples: 3000`: roughly 40–70 minutes for the
extraction, then a couple of minutes for everything else.

In [ ]:
# Point at the repo. Set exactly one of these.
REPO_URL = ""                       # e.g. "https://github.com/<user>/controlplane-cascade.git"
INPUT_DIR = "/kaggle/input/controlplane-cascade"   # used when REPO_URL is empty

import os, shutil, subprocess, sys
from pathlib import Path

WORK = Path("/kaggle/working/controlplane")
ACTIVATIONS = WORK / "results" / "activations.npz"

# Update in place when the repo is already here, rather than deleting and
# re-cloning. results/ is untracked, so `git reset --hard` leaves it alone --
# and results/activations.npz is 40-70 minutes of GPU time that a blind
# rmtree would throw away.
if WORK.exists() and (WORK / ".git").is_dir() and REPO_URL:
    print("repo already present; updating in place to preserve results/")
    subprocess.run(["git", "-C", str(WORK), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(WORK), "reset", "--hard", "FETCH_HEAD"], check=True)
elif REPO_URL:
    if WORK.exists():
        shutil.rmtree(WORK)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK)], check=True)
else:
    if WORK.exists():
        shutil.rmtree(WORK)
    shutil.copytree(INPUT_DIR, WORK)

os.chdir(WORK)
sys.path.insert(0, str(WORK))
print("working directory:", Path.cwd())
print("commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip())
print(sorted(p.name for p in Path.cwd().iterdir()))
print()
if ACTIVATIONS.is_file():
    print(f"activations.npz PRESENT ({ACTIVATIONS.stat().st_size / 1024**2:.0f} MB)"
          " -- you can re-run downstream stages with --from 02, no GPU hour needed")
else:
    print("activations.npz absent -- a full run is required (extraction included)")

In [ ]:
!pip install -q -r requirements.txt

## Stage 2 gate — does the model load, and does it answer?

TASKS.md Stage 2 asks for three things before the expensive stage: the model loads inside
the memory budget, one generated answer looks sane, and the resolved layer indices are
printed against the model's actual depth.

In [ ]:
import torch

from src.config import load_config, set_seeds, setup_logging
from src.model import describe_model, load_model_and_tokenizer, peak_memory_gb, sanity_generate

setup_logging()
config = load_config("config.yaml")
set_seeds(config.seed)

print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("config hash:", config.config_hash)

model, tokenizer = load_model_and_tokenizer(config)
info = describe_model(model, tokenizer, config)

print()
print("num_hidden_layers :", info["num_hidden_layers"])
print("hidden_size       :", info["hidden_size"])
print("probe layers      :", info["probe_layers"])
print("layer fractions   :", info["layer_fractions"])
print("padding side      :", info["padding_side"])
print("peak GPU memory   : %.2f GB" % peak_memory_gb())
print()
print("prompt template:")
print(info["example_prompt"])

In [ ]:
for question in [
    "Who wrote the novel 'Nineteen Eighty-Four'?",
    "What is the capital of Australia?",
    "Which element has the chemical symbol 'Au'?",
]:
    print(f"Q: {question}")
    print(f"A: {sanity_generate(model, tokenizer, question, config)!r}")
    print()

## Padding diagnostic — measure before committing to the full run

Prints what the equivalence check actually sees on this model: relative L2 error and
cosine similarity under left padding, then the same under deliberate right padding. The
gap between the two columns is the check's discriminating power on *your* hardware.

Takes about 20 seconds and needs no full run.

In [ ]:
import pandas as pd

from src.extract import compare_batched_unbatched, select_equivalence_prompts
from src.model import build_prompts, configure_tokenizer, resolve_layers
from src.data import prepare_dataset

frame, _ = prepare_dataset(config)
layers = resolve_layers(model, config)
prompts = select_equivalence_prompts(
    tokenizer,
    build_prompts(tokenizer, frame["question"].tolist(), config),
    config.equivalence_check.batch,
)
print("prompt token lengths:", [len(tokenizer(p)["input_ids"]) for p in prompts])

configure_tokenizer(tokenizer)
left = compare_batched_unbatched(model, tokenizer, prompts, layers)
tokenizer.padding_side = "right"
right = compare_batched_unbatched(model, tokenizer, prompts, layers)
configure_tokenizer(tokenizer)   # restore before anything else runs

rows = []
for layer in layers:
    key = str(layer)
    rows.append({
        "layer": layer,
        "LEFT rel L2": left["per_layer"][key]["max_relative_l2"],
        "LEFT cosine": left["per_layer"][key]["min_cosine"],
        "RIGHT rel L2": right["per_layer"][key]["max_relative_l2"],
        "RIGHT cosine": right["per_layer"][key]["min_cosine"],
        "activation norm": left["per_layer"][key]["reference_norm_median"],
    })
print()
print(pd.DataFrame(rows).to_string(index=False))
print()
print(f"limits: relative L2 <= {config.equivalence_check.relative_tolerance}, "
      f"cosine >= {config.equivalence_check.min_cosine}")
print("LEFT must pass both. RIGHT must fail both, by a wide margin.")

## Stage 3 pre-flight — do not skip this

Three checks before the GPU hour, per TASKS.md Stage 3:

1. the **left-padding equivalence check** on a batch of 4,
2. an **n=20 smoke run** whose completions you read by eye,
3. a **base-rate check** on those 20 — roughly half should be correct.

`--dry-run` writes nothing, so a bad result here costs a minute rather than an hour.
Free the notebook's model first so the subprocess gets the whole GPU.

In [ ]:
import gc

del model
gc.collect()
torch.cuda.empty_cache()

!python scripts/01_extract.py --config config.yaml --limit 20 --dry-run

**Read the output above before continuing.**

- The equivalence check reports **relative L2 error** and **cosine similarity**, not an
  absolute deviation — in bfloat16 the absolute number is dominated by rounding and means
  nothing on its own (DECISIONS.md 014). Expect relative L2 well under `0.10` and cosine
  above `0.999`.
- It also runs a **positive control**: the same comparison with the tokenizer deliberately
  right-padded, which must be *rejected*. You should see that line in the log. If the
  control passes, the run stops — the limits would not be discriminating anything.
- If the check failed it raised, and the padding is wrong — stop, do not work around it.
- The completions should be short answers, not echoes of the prompt or empty strings.
- Roughly half should be marked `OK`. `0/20` or `20/20` means the prompt or the matching
  rule is broken, not that the model is unusually bad or good.

## The full run

Extraction, probe, economics, latency, report.

**If `activations.npz` is already present** (the cell above says so), you do not need this
cell. Skip to the one below it and re-run only the downstream stages — a minute of CPU
instead of a couple of hours of GPU.

In [ ]:
!python scripts/run_all.py --config config.yaml

### Re-run downstream stages only

Use this when `activations.npz` already exists and only the probe, economics, latency or
report need regenerating — after a config change such as widening `probe.C_grid`.

Note that re-running stage 02 scores the test set again. That is recorded: every scoring
is appended to `results/test_scoring_log.json` and `RESULTS.md` discloses the count and
the full history (DECISIONS.md 016).

In [ ]:
!python scripts/run_all.py --config config.yaml --from 02

## The result

In [ ]:
from IPython.display import Markdown, display

display(Markdown(Path("results/RESULTS.md").read_text(encoding="utf-8")))

## Take the artifacts home

Everything a reviewer needs is small — the JSON files, the two plots, `RESULTS.md` and the
rendered `README.md`. The activations (~150 MB) and the parquet files stay behind; they are
regenerable and are gitignored.

Download `results_bundle.zip` from the Kaggle output pane, unzip it over `results/` in your
local checkout, then commit with an `exp:` message recording the numbers that moved.

In [ ]:
import zipfile

bundle = Path("/kaggle/working/results_bundle.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for pattern in ("results/*.json", "results/*.png", "results/RESULTS.md", "README.md"):
        for path in Path(".").glob(pattern):
            zf.write(path, path)
            print("added", path)
print()
print("wrote", bundle, f"({bundle.stat().st_size / 1024:.1f} KB)")